In [0]:
def transform_beneficiary(df):
    df.dropDuplicates(['BeneID'])
    return df


In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    current_date,
    lit,
    when, concat_ws
)
from functools import reduce
def validate_beneficary(df):
    chronic_columns=[
        "ChronicCond_Alzheimer",
        "ChronicCond_Heartfailure",
        "ChronicCond_KidneyDisease",
        "ChronicCond_Cancer",
        "ChronicCond_ObstrPulmonary",
        "ChronicCond_Depression",
        "ChronicCond_Diabetes",
        "ChronicCond_IschemicHeart",
        "ChronicCond_Osteoporasis",
        "ChronicCond_rheumatoidarthritis",
        "ChronicCond_stroke"
    ]

    invalid_condition=(
            (col("BeneID").isNull()) |
            (col("DOB").isNull()) |
            (col("DOB") > current_date()) |
            (
                (col("DOD").isNotNull()) &
                (col("DOD") < col("DOB"))
            ) |
            (~col("Gender").isin(1, 2)) |
            (~col("Race").isin(1, 2, 3, 5)) |
            (~col("RenalDiseaseIndicator").isin("0", "Y")) |
            (~col("NoOfMonths_PartACov").between(0, 12)) |
            (~col("NoOfMonths_PartBCov").between(0, 12)) |
            reduce(
                lambda x, y: x | y,
                [~col(c).isin(1, 2) for c in chronic_columns]
                )
    )

    reject_df=(df.filter(invalid_condition)\
    .withColumn(
    "reject_reason",
          concat_ws(
              "; ",
              when(col("BeneID").isNull(), "BeneID is NULL"),
              when(~col("Gender").isin(1,2), "Invalid Gender"),
              when(~col("Race").isin(1,2,3,5), "Invalid Race"),
              when(col("DOB") > current_date(), "Future DOB"),
              when(
                  col("DOD").isNotNull() &
                  (col("DOD") < col("DOB")),
                  "DOD before DOB"
              ),
              when(
                  ~col("NoOfMonths_PartACov").between(0,12),
                  "Invalid Part A Coverage"
              ),
              when(
                  ~col("NoOfMonths_PartBCov").between(0,12),
                  "Invalid Part B Coverage"
              )
          )
       )\
    .withColumn('ingestion_ts',current_timestamp())
    )

    valid_df=df.filter(~invalid_condition)
    return valid_df, reject_df
